# Instrument-Agnostic Automatic Music Transcription (Colab Inference)

This notebook allows you to run inference using the [Instrument-Agnostic AMT](https://github.com/anime-song/instrument-agnostic-amt) model.
It will automatically download the pre-trained model from Hugging Face.

## 1. Setup Environment

In [ ]:
# @title Install dependencies
!git clone https://github.com/anime-song/instrument-agnostic-amt.git
%cd instrument-agnostic-amt
!uv pip install pyyaml
!uv pip install einops
!uv pip install tqdm
!uv pip install soundfile
!uv pip install pretty_midi
!uv pip install mido
!uv pip install mir_eval
!uv pip install scipy
!uv pip install dlchordx
!uv pip install miditoolkit

!uv pip install stem-splitter
!uv pip install chord-romanizer

## 2. Prepare Audio

You can either upload a file or download one from a URL (e.g., YouTube).

In [ ]:
# @title Upload audio file
from google.colab import files
import os

uploaded = files.upload()
audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {audio_path}")

## 3. Stem Separation -> Transcribe Each Stem -> Instrument Refinement -> Merge -> Velocity Prediction -> Beat/Chord/Key Prediction

This section builds on the `batch_process_unlabeled.py` flow inside Colab.
It separates the uploaded song into stems, transcribes each stem, optionally relabels the instrument of every note with the instrument refinement model, merges the per-stem MIDI files into one result, predicts per-note velocity dynamics using separated stem audio, and optionally predicts beat, chord, and key information using the beat_chord model (`best_beat_chord_key.pth`).
Drum stems use the dedicated experimental `drums` model, bass stems use the `bass_v2` model, guitar stems use the `guitar_v1_5` model, and other stems use the `other_v1_5` model.
Instrument refinement (`REFINE_INSTRUMENTS`) listens to each separated stem again and reassigns the instrument class of its notes, so a stem whose notes were labeled with the wrong instrument can be corrected before merging. Drum and vocal stems are always skipped: drums have no non-drum candidate classes, and separating `melody` from `vocal_harmony` is a matter of musical role rather than timbre, which this model cannot judge.


In [ ]:
# @title Prepare stem-separated transcription helpers
from infer_stem import run_stem_separated_transcription


In [ ]:
# @title Run stem-separated transcription
OUTPUT_ROOT = "colab_outputs"  # @param {type:"string"}
WINDOW_BATCH_SIZE = 4  # @param {type:"integer"}
MAX_MIDI_MELODIC_INSTRUMENTS = 15  # @param {type:"integer"}
TRANSCRIBE_DRUM_STEMS = True  # @param {type:"boolean"}
REFINE_INSTRUMENTS = False  # @param {type:"boolean"}
REFINEMENT_CHECKPOINT = ""  # @param {type:"string"}
REFINEMENT_MODE = "cluster"  # @param ["cluster", "single"]
PREDICT_VELOCITY = True  # @param {type:"boolean"}
PREDICT_BEAT_CHORD = False  # @param {type:"boolean"}
CLEANUP_SEPARATED_STEMS = False  # @param {type:"boolean"}
MERGE_ONSET_MS = 50.0  # @param {type:"number"}
LOW_VRAM_MODE = False  # @param {type:"boolean"} Keep models in CPU RAM, move one to GPU per stem
NO_HALF = False  # @param {type:"boolean"} Disable fp16 (e.g. GTX 16-series): fp32 separation with halved chunks

if "audio_path" not in globals():
    raise RuntimeError("Please upload an audio file first.")

stem_pipeline_result = run_stem_separated_transcription(
    audio_path,
    checkpoint_path=None,
    output_root=OUTPUT_ROOT,
    window_batch_size=WINDOW_BATCH_SIZE,
    max_midi_melodic_instruments=MAX_MIDI_MELODIC_INSTRUMENTS,
    transcribe_drum_stems=TRANSCRIBE_DRUM_STEMS,
    refine_instruments=REFINE_INSTRUMENTS,
    refinement_checkpoint_path=REFINEMENT_CHECKPOINT or None,
    refinement_mode=REFINEMENT_MODE,
    predict_velocity=PREDICT_VELOCITY,
    predict_beat_chord=PREDICT_BEAT_CHORD,
    cleanup_separated_stems=CLEANUP_SEPARATED_STEMS,
    merge_onset_ms=MERGE_ONSET_MS,
    low_vram_mode=LOW_VRAM_MODE,
    no_half=NO_HALF,
)
stem_pipeline_result


In [ ]:
# @title Download stem-separated results
from google.colab import files
from pathlib import Path
import shutil

if "stem_pipeline_result" not in globals():
    print("Run the stem-separated transcription cell first.")
else:
    merged_midi_path = Path(stem_pipeline_result["merged_midi_path"])
    stem_midi_dir = Path(stem_pipeline_result["stem_midi_dir"])
    zip_base = stem_midi_dir.parent / f"{stem_midi_dir.name}"
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=stem_midi_dir))

    print(f"Downloading merged MIDI: {merged_midi_path}")
    files.download(str(merged_midi_path))


## Optional: Run Inference

In [ ]:
# @title Run Transcription
!python infer.py --audio "{audio_path}"

import os
midi_path = os.path.splitext(audio_path)[0] + ".mid"
if os.path.exists(midi_path):
    print(f"Success! MIDI saved to: {midi_path}")
else:
    print("Error: MIDI file was not generated.")

## Optional: Download Results

In [ ]:
# @title Download MIDI file
if os.path.exists(midi_path):
    files.download(midi_path)
else:
    print("No MIDI file to download.")